In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

In [3]:
# ============================================
# 2. Load dataset
# ============================================

df = pd.read_csv("Customer-Churn.csv")

print("First 5 rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

First 5 rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies       

In [4]:
# ============================================
# 3. Clean column names
# ============================================

df.columns = df.columns.str.strip()

In [5]:
# ============================================
# 4. Remove unnecessary ID columns
# ============================================

for col in ["CustomerID", "customerID", "customer_id", "id", "ID"]:
    if col in df.columns:
        df = df.drop(col, axis=1)

In [6]:
# ============================================
# 5. Convert TotalCharges if available
# ============================================

if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"], errors="coerce"
    )

In [7]:
# ============================================
# 6. Remove missing values
# ============================================

df = df.dropna()

print("\nMissing values after cleaning:")
print(df.isnull().sum().sum())


Missing values after cleaning:
0


In [8]:
# ============================================
# 7. Convert categorical columns to numbers
# ============================================

# Convert Yes/No type columns
for col in df.select_dtypes(include="object").columns:

    if col != "Churn":
        df[col] = df[col].astype("category").cat.codes


# Convert target column
if df["Churn"].dtype == "object":
    df["Churn"] = df["Churn"].map({
        "Yes": 1,
        "No": 0
    })

C:\Users\Arati\AppData\Local\Temp\ipykernel_3848\2925099703.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [9]:
# ============================================
# 8. Create X and y
# ============================================

X = df.drop("Churn", axis=1)
y = df["Churn"]


print("\nX shape:", X.shape)
print("y shape:", y.shape)


X shape: (7032, 19)
y shape: (7032,)


In [10]:
# ============================================
# 9. Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
# ============================================
# 10. Scale the data
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Scale complete data for Cross Validation
X_scaled = scaler.fit_transform(X)

In [12]:
# ============================================
# 11. Create 4 models
# ============================================

models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            random_state=42
        )
}

In [13]:
# ============================================
# 12. Train and Compare Models
# ============================================

results = []

for name, model in models.items():

    # Train model
    model.fit(X_train_scaled, y_train)

    # Test prediction
    y_pred = model.predict(X_test_scaled)

    # Test accuracy
    test_accuracy = accuracy_score(y_test, y_pred)

    # 5-Fold Cross Validation
    cv_scores = cross_val_score(
        model,
        X_scaled,
        y,
        cv=5,
        scoring="accuracy"
    )

    cv_accuracy = cv_scores.mean()

    results.append({
        "Model": name,
        "Test Accuracy": round(test_accuracy, 4),
        "CV Accuracy": round(cv_accuracy, 4)
    })

In [14]:
# ============================================
# 13. Create Comparison Table
# ============================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="CV Accuracy",
    ascending=False
)

print("\n====================================")
print("MODEL COMPARISON")
print("====================================")

print(results_df.to_string(index=False))


MODEL COMPARISON
              Model  Test Accuracy  CV Accuracy
  Gradient Boosting         0.7974       0.8042
Logistic Regression         0.7939       0.8012
      Random Forest         0.7818       0.7873
      Decision Tree         0.7143       0.7271


In [15]:
# ============================================
# 14. Select Winner
# ============================================

winner = results_df.iloc[0]

print("\n====================================")
print("WINNER")
print("====================================")

print("Best Model:", winner["Model"])
print("CV Accuracy:", winner["CV Accuracy"])


WINNER
Best Model: Gradient Boosting
CV Accuracy: 0.8042


In [16]:
# ============================================
# 15. Reason
# ============================================

print("\nReason:")
print(
    winner["Model"],
    "is the winner because it achieved the highest "
    "Cross-Validation Accuracy among the four models."
)


Reason:
Gradient Boosting is the winner because it achieved the highest Cross-Validation Accuracy among the four models.
